# Notebook 22 — E5: Training-only diagnosis of SCV-context benefits and rare attacks

**Why this experiment?** Our frozen 444-feature model achieved **86.3218% official accuracy and 58.1397% official macro-F1**, numerically exceeding SCV-IDS's reported 85.77% accuracy and 46.87% macro-F1. The *official test has already been consumed*. **E5 NEVER reopens the official test or its stored per-example predictions**, and does not train or tune a new model.

We must answer whether window adjacency is meaningful or partly an ordering artifact, and identify what makes Analysis, Backdoor, and DoS difficult *without inspecting their official test errors again*.

## E5: seven evidence-based diagnostic sections

1. **CSV ordering audit:** Analyze source class transitions, true class-run lengths, observed adjacent same-label fraction versus chance rate `sum(p_class**2)`, independently in the full original training CSV and the frozen development train/validation partitions. The goal is to detect sorted-label/run artifacts, NOT to establish actual time.
2. **Window purity:** Fraction of five-record windows whose four historical records share the target label, separately by class and partition.
3. **Original source-index gaps:** Determine how often positions in the original CSV remain contiguous after the original internal group-disjoint split. Record gap distributions. No timestamps are inferred.
4. **Duplicate-group integrity:** Reconstruct identical-feature groups solely from the original training file and verify none cross the frozen train/validation boundary. Count groups with conflicting labels.
5. **Real rare-attack errors:** Reuse the **unaltered E3 fitted model and its actual private internal-validation probabilities** (35,005 targets). Export aggregate class confusion, false-negative breakdown, per-class confidence, ten-bin expected calibration error and multiclass Brier score; exclude per-row IDs and probabilities from public GitHub.
6. **Feature importance and one-lag perturbations:** Audit E3 LightGBM's genuine 444 fitted feature gains. For each of historical positions 1–4, shuffle only that whole 42-field record on validation windows and recompute all dependent statistics. Repeat with three fixed seeds. **Same original fitted model, no retraining;** newest target remains unchanged. Interpret as intervention under feature-distribution shift, not causal proof.
7. **Paired internal-validation effect sizes:** Compare frozen E2 and E3 validation decisions. An exploratory 64-target block bootstrap quantifies variability while acknowledging overlapping windows and that the same validation partition was previously used for early stopping.

## Input integrity and privacy

- Only source file opened: original `UNSW_NB15_training-set.csv`, SHA256 `bec7dd5e...`.
- Frozen SCV-IDS train-only group-disjoint split and validation targets hash-verified.
- Original E2 window-rare model and E3 stats-rare model/checkpoints restored **privately** from unpublished GitHub draft releases; strictly read-only, verified SHA256.
- No official test CSV, official test predictions, new models, official-test retuning or model changes.
- Public GitHub receives **only allowlisted aggregated research reports**. No source rows, per-example labels/probabilities, fitted model, or secret.
- **CSV order has not been verified to be chronology**. Results must not be presented as true temporal effects.

## Kaggle settings

**Accelerator None (CPU)**, Internet **ON**, attach `dsshayan/unsw-nb15` and authorize existing secret `pushRare`. Run the **single code cell** below, ideally **Save Version → Save & Run All**. Do not rerun Notebook 20. Approximate cost is a few minutes on Kaggle CPU because there is no fitting; actual time varies.

When E5 finishes, look for `E5 TRAIN-ONLY AUDIT COMPLETE — NO NEW MODELS, NO OFFICIAL TEST` and a GitHub results link. Use that report to decide whether a new method is scientifically justified before further training.

**Pre-registered report:** `configs/e5_context_diagnostic_001.json`.

In [ ]:
# ONE CELL: frozen internal training-only context/label-order audit (NO official test).
import os,sys,hashlib,subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-e5")
BASE=Path("/kaggle/working/research_outputs")
E2=BASE/"WINDOW-E2-001-S42"
E3=BASE/"WINDOW-E3-001-S42"
OUT=BASE/"E5-CONTEXT-DIAGNOSTIC-001"
SOURCE=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert SOURCE.is_file(), "Attach the original dsshayan/unsw-nb15 source dataset"
with SOURCE.open("rb") as stream:
    assert hashlib.file_digest(stream,"sha256").hexdigest()==(
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
    ), "TRAIN-ONLY dataset hash mismatch; do not continue with different inputs"
token=UserSecretsClient().get_secret("pushRare")
assert token, "Enable the existing pushRare GitHub secret in this Kaggle notebook"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
env=os.environ.copy()
env["GITHUB_TOKEN"]=token
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
print("E5 SOURCE CODE:",
    subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                            text=True).strip(),flush=True)

# Existing frozen original validation split; no training and no test data.
for script,archive in [
    ("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar")
]:
    subprocess.run([sys.executable,str(REPO/"scripts"/script),
        "--base",str(BASE),"--archive",str(Path("/kaggle/working")/archive)],
        cwd=str(REPO),env=env,check=True)

sys.path.insert(0,str(REPO/"scripts"))
from archive_window_e2 import restore_prior_candidate as e2_restore,validate as e2_verify
from archive_window_e3 import restore_prior_candidate as e3_restore,validate as e3_verify
e2_restore(E2,"window_rare_sqrt",token)
e2_verify(E2,"window_rare_sqrt")
e3_restore(E3,"stats_rare_sqrt",token)
e3_verify(E3,"stats_rare_sqrt")
print("Frozen E2 and E3 models restored for READ-ONLY diagnostics.",flush=True)

subprocess.run([
    sys.executable,str(REPO/"scripts/run_e5_context_diagnostic.py"),
    "--repo",str(REPO),
    "--config",str(REPO/"configs/e5_context_diagnostic_001.json"),
    "--train-csv",str(SOURCE),
    "--reference",str(BASE/"SCV-IDS-REPRO-001"),
    "--e2",str(E2),"--e3",str(E3),
    "--output",str(OUT),"--sync"
],cwd=str(REPO),env=env,check=True)

print("\nE5 FINAL SUMMARY:",flush=True)
print((OUT/"summary.json").read_text(),flush=True)
print("Read-only E5 completed; neither model training nor official testing occurred.",flush=True)
